## Setup

### Load Modules

In [ ]:
%load_ext autoreload
%autoreload 2

#General Import
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle
from os.path import join
from sklearn.model_selection import KFold
from matplotlib import colormaps as cmaps
from mne.filter import filter_data, resample
import scipy.stats as stats
import pandas as pd
from itertools import product
import xarray as xr
from scipy.signal import coherence, welch

#ML Import
from sklearn.decomposition import PCA, FastICA, SparsePCA, FactorAnalysis
from sklearn.preprocessing import StandardScaler, scale
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import jPCA
import scipy.signal as signal
from statsmodels.tsa.stattools import grangercausalitytests
import tslearn
from dtaidistance import dtw
import dtaidistance as dta
from dtaidistance.preprocessing import differencing
import dtaidistance.clustering.kmeans as dtwkmeans


#Electrophysiology Import
from spyeeg.models.TRF import TRFEstimator
from spyeeg.models.ERP import ERP_class
from spyeeg.utils import lag_matrix
import mne
import frites
from frites.simulations import sim_multi_suj_ephy
from frites.dataset import DatasetEphy
from frites.workflow import WfConnComod
from frites import set_mpl_style
import frites.conn as conn
from scipy.signal import welch
import spectral_connectivity 

#Graph Import
import networkx as nx


#Performance Import
import time
import psutil

#Local Import
from stats_utils import cliffs_delta, cohen_d
from nice_utils import estimate_loop_time, decorator_loop
from preprocessing_utils import mono_to_bipolar, select_channels, available_regions, delete_channels, adj_scale
from signal_utils import sparse_resample, lag_finder, onmf
from viz_utils import create_matshow_gif, create_collection_gif, _arrow3D

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load Features

In [ ]:
# Acoustic Regressors
fs = 100
new_path = 'C:/Users/D-CAP/Documents/GitHub/witching-star/regressors/selected_regs.pkl'
new_data = pickle.load(open(new_path, 'rb'))
data_fs = new_data['fs']
new_regressors = new_data['regs']
new_names = new_data['regs_name']
ratio = fs/data_fs
new_duration = int(new_regressors.shape[0] * ratio) + 1

new_resamp = []
for i in range(new_regressors.shape[1]):
    name = new_names[i]
    if name in ['Intensity', 'Envelope Oganian', 'Envelope Derivative TF', 'F0 Loudness', 'SpectralFlux Filtered', 'SpectralFlux not_filtered']:
        new_reg = mne.filter.resample(new_regressors[:,i], up=100, down=data_fs)[:new_duration]
    elif name in ['peakEnv_tf', 'Syllabe Onset', 'p-syl', 'Phono']:
        new_reg = new_regressors[:,i] - np.min(new_regressors[:,i])
        new_reg = sparse_resample(new_reg, new_fs = fs, current_fs = data_fs)[:new_duration]
    else:
        print('wut')
    new_resamp.append(new_reg)
new_resamp = np.asarray(new_resamp).T
regressors = new_resamp
regressors_name = new_data['regs_name']


In [ ]:
# Renyi2 Array

path_renyi = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/renyi_array2.pickle"
renyi_data = pickle.load(open(path_renyi, 'rb'))
data_fs = renyi_data['fs']
X = np.roll(renyi_data['X'],4, axis=0)
onsets = np.where(X[:,5] >0)[0]
reg_renyi = np.zeros([regressors.shape[0], X.shape[1]])

for onset in onsets:
    new_onset = int(onset/data_fs*fs)
    for renyi_index in range(X.shape[1]):
        reg_renyi[new_onset, renyi_index] = X[onset,renyi_index]

regressors = np.hstack([regressors, reg_renyi])
regressors_name = regressors_name + renyi_data['names']
renyi_values_wrd = [float(renyi_name.split('renyi ')[1]) for renyi_name in renyi_data['names'][:20]]

### Load Neural Data

In [ ]:
# Load Broadband

data_subject = dict()
channels_subject = dict()
locations_subject = dict()
path_data = "D:/DataSEEG_Sorciere/BIDS/data_mne_fif"
for index_subject in range(1,40):
    try:
        name_subject = 'sub-{:03}'.format(index_subject)
        name_file = name_subject + '_task-iSpeech_speech-epo.fif'
        path_file = os.path.join(path_data, name_subject,'preprocessed','epochs',name_file)
        if os.path.isfile(path_file):
            mne_data = mne.read_epochs(path_file, verbose = False)
        elif os.path.isfile(os.path.join(path_data, name_subject,'preprocessed','monopolar',name_file)):
            path_file = os.path.join(path_data, name_subject,'preprocessed','monopolar',name_file)
            mne_data = mne.read_epochs(path_file, verbose = False)
        else:
            path_file = os.path.join(path_data, name_subject,'preprocessed','epochs','monopolar',name_file)
            mne_data = mne.read_epochs(path_file, verbose = False)
        mne_data_resample = mne_data.resample(fs, npad = 'auto', verbose = False)
        mne_data_resample.filter(0.3, 49, verbose = False)
        channels = mne_data_resample.ch_names
        montage = mne_data_resample.get_montage()
        ch_names = [n for n,_ in montage.get_positions()['ch_pos'].items()] 
        loc = (1e3 * np.stack([coord for _,coord in montage.get_positions()['ch_pos'].items()])).T  # store locations
        
        data_subject[index_subject] = mne_data_resample.get_data(copy = False)[0].T[:regressors.shape[0],:]
        channels_subject[index_subject] = channels
        locations_subject[index_subject] = loc
        print('Subject', index_subject, 'loaded')
    except:
        print('Error in subject', index_subject)

data_bipolar, channels_bipolar, locations_bipolar = mono_to_bipolar(data_subject, channels_subject, locations_subject)

In [ ]:
atlas_subject = dict()
path_data = "D:/DataSEEG_Sorciere/BIDS/data_mne_fif"
path_atlas = "D:/DataSEEG_Sorciere/BIDS/freesurfer"
for index_subject in range(1,40):
    try:
        name_subject = 'sub-{:03}'.format(index_subject)
        name_bipolar_atlas = 'elecbipolar2atlas.mat'
        name_monopolar_atlas = 'elec2atlas.mat'
        path_bipolar_atlas = os.path.join(path_atlas, name_subject,name_bipolar_atlas)
        path_monopolar_atlas = os.path.join(path_atlas, name_subject,name_monopolar_atlas)
        bipolar_atlas = get_bipolar_atlas(path_bipolar_atlas, atlas = 'Desikan_Killiany') #Destrieux
        atlas_subject[index_subject] = bipolar_atlas
    except:
        print('Atlas error in subject', index_subject)

subjects =  list(atlas_subject.keys())
new_atlas = dict()
for subject_index in atlas_subject:
    new_atlas[subject_index] = dict()
    for channels_bipolar in atlas_subject[subject_index]:
        new_name = channels_bipolar.split('-')[0] + '||' + channels_bipolar.split('-')[1]
        new_atlas[subject_index][new_name] = atlas_subject[subject_index][channels_bipolar][0]

## Compute PID

In [ ]:
exclude = True
montage_choice = 'bipo'
mi_type = 'cc'
channel_selection = []
channel_selection_join = ''.join(channel_selection)
tmin = -1 #-0.5 -2
tmax = 1.3 #1.3 1.3
step = 1
env_ref = 1
baseline_limits = [0,50]
moving_avg = 3
apply_baseline = True
baseline_str = (not apply_baseline) * 'no_baseline'
time_array = np.linspace(tmin, tmax,int((tmax-tmin)*fs))

for regressor1, regressor2 in zip([233,233,253],[253,258,258]):
    pid_data = dict()
    pid_channels = dict()
    for subject_index in range(len(data_subject)):
        subject_id = list(data_subject.keys())[subject_index]
        tot_list, unique_list, red_list, syn_list = [], [], [], []
        eeg_mono = data_subject[subject_id]
        channels_mono = channels_subject[subject_id]
        eeg_bipo = data_bipolar[subject_id]
        channels_bipo = channels_bipolar[subject_id]
        eeg_mono_HT, channels_mono_HT = select_channels(eeg_mono,channels_mono, channel_select = channel_selection, exclude = exclude)
        eeg_bipo_HT, channels_bipo_HT = select_channels(eeg_bipo,channels_bipo, channel_select = channel_selection, exclude = exclude)
        y1_mono = eeg_mono_HT[:,:]
        y1_bipo = eeg_bipo_HT[:,:]
        
        if montage_choice == 'ica':
            y_ica, channels_ica = ica_shaft(y1_mono, channels_mono_HT, random_state = 0)
            channels_name = channels_ica
            y = y_ica
        elif montage_choice == 'mono':
            channels_name = channels_mono_HT 
            y = y1_mono
        elif montage_choice == 'bipo':
            channels_name = []
            for chan_name in channels_bipo_HT:
                channels_name.append(chan_name.replace('-', '||'))
            y = y1_bipo

        x1 = regressors[:y.shape[0],regressor1]
        x2 = regressors[:y.shape[0],regressor2]
        x2[x1 == 0] = 0 
        x1,x2 = sparse_realign(x1,x2)
        x1,x2 = common_events(x1, x2)
        signal1 = x1
        signal2 = x2
        erp1 = ERP_class(tmin = tmin, tmax = tmax, srate=100)
        erp1.add_events(y, signal1, weight_events = False, record_weight = True)
        erp2 = ERP_class(tmin = tmin, tmax = tmax, srate=100)
        erp2.add_events(y, signal2, weight_events = False, record_weight = True)
        epoched_data1 = np.asarray(erp1.evoked).transpose(0,2,1)
        epoched_data2 = np.asarray(erp2.evoked).transpose(0,2,1)
        if apply_baseline:
            baseline1 = np.repeat(epoched_data1[:,:,baseline_limits].mean(-1), epoched_data1.shape[-1]).reshape(epoched_data1.shape)
            epoched_data1 = epoched_data1 - baseline1
            baseline2 = np.repeat(epoched_data2[:,:,baseline_limits].mean(-1), epoched_data2.shape[-1]).reshape(epoched_data2.shape)
            epoched_data2 = epoched_data2 - baseline2
        epoched_reg1 = np.asarray(erp1.weights)
        epoched_reg2 = np.asarray(erp2.weights)
        epoched_reg1 = epoched_reg1[:min(len(epoched_reg1), len(epoched_reg2))]
        epoched_reg2 = epoched_reg2[:min(len(epoched_reg1), len(epoched_reg2))]
        epoched_data1 = epoched_data1[:min(len(epoched_reg1), len(epoched_reg2))]
        epoched_data2 = epoched_data2[:min(len(epoched_reg1), len(epoched_reg2))]
        epoched_reg = np.vstack([epoched_reg1, epoched_reg2]).T[:,:,np.newaxis]

        print(subject_id, y.shape)
        for t in range(len(time_array) - moving_avg):
            tot_channel, unique_channel, red_channel, syn_channel = [], [], [], []
            epoched_data1_t = epoched_data1[:,:,t:t+moving_avg].mean(-1)
            epoched_data2_t = epoched_data2[:,:,t:t+moving_avg].mean(-1)
            for channel in range(y.shape[1]):
                epoched_data_t_roi = epoched_data1_t[:,channel]
                ii = conn.conn_pid(epoched_reg, epoched_data_t_roi, 
                                   roi=[regressors_name[regressor1], regressors_name[regressor2]], 
                                   times=time_array[t:t+moving_avg], 
                                   mi_type=mi_type, gcrn=True, dt=1, verbose =False)
                tot_channel.append(ii[0].data[0,0]), unique_channel.append(ii[1].data[:,0]), red_channel.append(ii[2].data[0,0]), syn_channel.append(ii[3].data[0,0])
            tot_list.append(tot_channel), unique_list.append(unique_channel), red_list.append(red_channel), syn_list.append(syn_channel)
        pid_data[subject_id] = {'total': np.asarray(tot_list),
                        'unique': np.asarray(unique_list),
                        'redundancy': np.asarray(red_list),
                        'synergy': np.asarray(syn_list)}
        pid_channels[subject_id] = channels_name

    pid_info = {'regressor_index 1': regressor1, 'regressor_index 2': regressor2,
            'regressor 1': regressors_name[regressor1], 'regressor 2': regressors_name[regressor2], 
            'timearray' : time_array[moving_avg//2 + moving_avg%2:-moving_avg//2+ moving_avg%2], 'fs' : fs,
            'data': pid_data, 'channels': pid_channels}
    filename = 'PID_reg/PID6_' + baseline_str + montage_choice + '_' + channel_selection_join + '_regpair_' + str(regressor1) + '-' + str(regressor2) + '.pickle'
    with open(filename, 'wb') as file:
        pickle.dump(pid_info, file)
